In [57]:
pip install --quiet nltk scikit-learn PyPDF2


Note: you may need to restart the kernel to use updated packages.


In [58]:
import re
import json

from pathlib import Path
from pprint import pprint

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [59]:
BASE = Path.cwd().parent
DATA_DIR = BASE / "data"
OUTPUTS_DIR = BASE / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

JOB_PATH = DATA_DIR / "job_description.txt"
RESUME_PATH = DATA_DIR / "SarthakResume.pdf"
RESULTS_PATH = OUTPUTS_DIR / "results.txt"

print("Base:", BASE)
print("Data dir:", DATA_DIR)
print("Job desc file exists:", JOB_PATH.exists())
print("Resume PDF exists:", RESUME_PATH.exists())

Base: c:\Users\sarth\Desktop\Resume_matcher
Data dir: c:\Users\sarth\Desktop\Resume_matcher\data
Job desc file exists: True
Resume PDF exists: True


In [60]:
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text
def extract_email(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    match = re.search(email_pattern, text)
    return match.group(0) if match else None
def extract_phone(text):
    phone_pattern = r'(\+?\d{1,3}[-.\s]?)?(\(?\d{3}\)?[-.\s]?)?\d{3}[-.\s]?\d{4}'
    match = re.search(phone_pattern, text)
    return match.group(0) if match else None

In [61]:
if not JOB_PATH.exists():
    # create a sample job description if missing
    JOB_PATH.write_text(
        "Looking for NLP engineer with Python, Machine Learning, TensorFlow, "
        "text preprocessing, NLTK, and basic SQL skills."
    )

job_text = JOB_PATH.read_text(encoding='utf-8')
print("Job description (preview):\n", job_text[:400])


Job description (preview):
 We are seeking a motivated Software Development Engineer with strong expertise in backend development, AI/ML, and data-driven applications. The ideal candidate will have experience building scalable backend systems using Django and REST APIs, implementing JWT authentication and role-based permissions, and integrating AI/ML models for production use. Responsibilities include designing and optimizin


In [62]:
if not RESUME_PATH.exists():
    raise FileNotFoundError(f"Put candidate resume PDF at: {RESUME_PATH}")

reader = PdfReader(str(RESUME_PATH))
resume_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume_text += text + "\n"

print("Resume text preview:\n")
print(resume_text)


Resume text preview:

     SUMMARY
2020 - 2024 B.Tech
Biju Patnaik University of Technology 
Department of Computer Science and Engineering. Equivalent CGPA: 8.27 CGPA
Mar 2024 – Aug 2024 Numetry Technologies: Software Development Trainee 
Developed scalable backend applications using Django, implementing REST APIs, JWT
authentication, and role-based permissions. Built reusable modules and proof-of-concept AI/ML
integrations for faster development and data-driven features.Motivated Computer Science graduate with strong expertise in Data Science, AI/ML, and backend
development. Skilled in building scalable data-driven applications, predictive models, and RESTful APIs.
Experienced in Python programming, data analytics, visualization, and machine learning. Quick learner with
strong problem-solving abilities, capable of contributing independently or in team environments.SARTHAK RANJAN MISHRA 
Bengaluru, Karnataka |  sarthakranjan1359@gmail.com | https://github.com/sarthak-56 | +91 70083921

In [63]:
email = extract_email(resume_text)
phone = extract_phone(resume_text)
print("Detected email:", email)
print("Detected phone:", phone)


Detected email: sarthakranjan1359@gmail.com
Detected phone: +91 7008392169


In [64]:
job_clean = clean_text(job_text)
resume_clean = clean_text(resume_text)

print("Cleaned job preview:", job_clean[:200])
print("Cleaned resume preview:", resume_clean[:200])


Cleaned job preview: seeking motivated software development engineer strong expertise backend development aiml datadriven applications ideal candidate experience building scalable backend systems using django rest apis im
Cleaned resume preview: summary btech biju patnaik university technology department computer science engineering equivalent cgpa cgpa mar aug numetry technologies software development trainee developed scalable backend appli


In [65]:
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([resume_clean, job_clean])

similarity_score = 0.0
if vectors.shape[0] == 2:
    similarity_score = cosine_similarity(vectors[0], vectors[1])[0][0]

similarity_percent = round(similarity_score * 100, 2)
print(f"Resume Match Similarity: {similarity_percent} %")


Resume Match Similarity: 47.1 %


In [66]:
job_features = vectorizer.get_feature_names_out()
resume_tokens = set(resume_clean.split())

found = [w for w in job_features if w in resume_tokens]
missing = [w for w in job_features if w not in resume_tokens]

print("Keywords found in resume (from job features):")
pprint(found)
print("\nKeywords missing (from job features):")
pprint(missing)


Keywords found in resume (from job features):
['abilities',
 'actionable',
 'admin',
 'agile',
 'aiml',
 'algorithms',
 'analysis',
 'analytics',
 'api',
 'apis',
 'applications',
 'aug',
 'authentication',
 'aws',
 'backend',
 'banking',
 'basics',
 'bayes',
 'bengaluru',
 'biju',
 'btech',
 'building',
 'built',
 'business',
 'capable',
 'cart',
 'cgpa',
 'checkout',
 'churn',
 'cicd',
 'classification',
 'classify',
 'cleaning',
 'cloud',
 'cloudinary',
 'complete',
 'computer',
 'contributing',
 'created',
 'css',
 'customer',
 'data',
 'databases',
 'datadriven',
 'deep',
 'department',
 'deposits',
 'developed',
 'development',
 'django',
 'ec',
 'ecommerce',
 'ecr',
 'eda',
 'educationprogramming',
 'engineer',
 'engineering',
 'enhanced',
 'environmentssarthak',
 'equivalent',
 'experience',
 'experienced',
 'expertise',
 'faster',
 'feature',
 'featuresmotivated',
 'featuresoptimized',
 'followlike',
 'forest',
 'frontend',
 'full',
 'fullstack',
 'fund',
 'generate',
 'gradua

In [67]:
results = {
    "similarity_percent": similarity_percent,
    "email": email,
    "phone": phone,
    "keywords_found": found,
    "keywords_missing": missing,
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    f.write(json.dumps(results, indent=2))

print("Results written to:", RESULTS_PATH)
print(json.dumps(results, indent=2))


Results written to: c:\Users\sarth\Desktop\Resume_matcher\outputs\results.txt
{
  "similarity_percent": 47.1,
  "email": "sarthakranjan1359@gmail.com",
  "phone": "+91 7008392169",
  "keywords_found": [
    "abilities",
    "actionable",
    "admin",
    "agile",
    "aiml",
    "algorithms",
    "analysis",
    "analytics",
    "api",
    "apis",
    "applications",
    "aug",
    "authentication",
    "aws",
    "backend",
    "banking",
    "basics",
    "bayes",
    "bengaluru",
    "biju",
    "btech",
    "building",
    "built",
    "business",
    "capable",
    "cart",
    "cgpa",
    "checkout",
    "churn",
    "cicd",
    "classification",
    "classify",
    "cleaning",
    "cloud",
    "cloudinary",
    "complete",
    "computer",
    "contributing",
    "created",
    "css",
    "customer",
    "data",
    "databases",
    "datadriven",
    "deep",
    "department",
    "deposits",
    "developed",
    "development",
    "django",
    "ec",
    "ecommerce",
    "ecr",
  

In [69]:
from IPython.display import display, Markdown

md = f"""
# Resume Match Report

**Similarity:** **{similarity_percent}%**

**Email:** {email or 'Not found'}  
**Phone:** {phone or 'Not found'}

**Keywords found:** {', '.join(found) if found else 'None'}  

**Keywords missing:** {', '.join(missing) if missing else 'None'}  
"""

# Display in notebook
display(Markdown(md))



# Resume Match Report

**Similarity:** **47.1%**

**Email:** sarthakranjan1359@gmail.com  
**Phone:** +91 7008392169

**Keywords found:** abilities, actionable, admin, agile, aiml, algorithms, analysis, analytics, api, apis, applications, aug, authentication, aws, backend, banking, basics, bayes, bengaluru, biju, btech, building, built, business, capable, cart, cgpa, checkout, churn, cicd, classification, classify, cleaning, cloud, cloudinary, complete, computer, contributing, created, css, customer, data, databases, datadriven, deep, department, deposits, developed, development, django, ec, ecommerce, ecr, eda, educationprogramming, engineer, engineering, enhanced, environmentssarthak, equivalent, experience, experienced, expertise, faster, feature, featuresmotivated, featuresoptimized, followlike, forest, frontend, full, fullstack, fund, generate, graduate, history, html, httpsgithubcomsarthak, httpswwwlinkedincominsarthakranjanmishraa, iam, idf, implemented, implementing, including, independently, insightsprojects, integration, integrations, interaction, jwt, karnataka, learner, learning, like, listing, llm, logistic, machine, management, mar, matplotlib, media, microservices, mishra, ml, model, models, modules, mongodb, movie, mysql, naive, native, negative, nextjs, nlp, nltk, numetry, numpy, optimized, orders, others, pandas, patnaik, performance, performed, permissions, platform, positive, postgresql, posts, predicted, prediction, predictive, problemsolving, product, professional, profiles, programming, proofofconcept, python, quick, rag, random, ranjan, rds, react, reactjs, realtime, regression, responsive, rest, restful, reusable, review, reviews, rolebased, sarthakranjangmailcom, scalable, science, scikitlearn, seaborn, secure, sentiment, server, shopping, skilled, skills, smooth, social, software, sql, strong, structures, summary, supporting, system, tailwind, team, technical, technologies, technology, testing, tf, tools, trainee, transaction, transfers, unit, university, user, using, visualization, warehousing, withdrawals, workflow  

**Keywords missing:** ability, backends, basic, candidate, collaborating, deploying, deployment, designing, developing, essential, exploratory, exposure, familiarity, ideal, include, integrating, motivated, optimizing, performing, platforms, plus, preprocessing, production, required, responsibilities, seeking, services, systems, teams, use, work  


In [70]:
from pathlib import Path

OUTPUTS_DIR = Path("../outputs")  # or your preferred folder
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

md_file = OUTPUTS_DIR / "resume_match_report.md"

with open(md_file, "w", encoding="utf-8") as f:
    f.write(md)

print(f"Markdown report saved at: {md_file}")


Markdown report saved at: ..\outputs\resume_match_report.md
